# Explanation disagreement: shared-reference mitigation (M1) and realistic deletion-faithfulness — BDHS 2022 child stunting

**Version m1-v1** (specification v2 of 19 Oct 2026, frozen before the first M1 run). Built on the validated v3.1.1 protocol code (data, folds, models, explainers and agreement metrics are used unchanged).

**Question (RQ3):** if SHAP, LIME and Integrated Gradients are given the *same reference distribution*, does their child-specific agreement increase, and do the explanations stay faithful to the model?

| Arm | SHAP | LIME | IG |
|---|---|---|---|
| **M0** default (recomputed, checked against the FULL run) | 50-row background (main-run draw) | full training fold | training mean of encoded inputs |
| **M1** shared | 50-row reference B | `training_data` = B | IG averaged over the 50 real baselines in B |
| **C1** control | B | independent 50 rows B' | IG over independent 50 rows B'' |

**Method–model compatibility (unchanged):** LR: SHAP, LIME, IG (float64 PyTorch twin); RF: SHAP, LIME (IG not applicable); MLP: SHAP, LIME, IG.
**Primary outcome:** Δ = (C − N2)[M1] − (C − N2)[M0], Spearman on |attribution|, 7 model×pair cells, paired bootstrap over explained children. **Faithfulness:** deletion with *real-child replacements* (conditional on the kept variables; marginal as a secondary scheme) against random deletion order.

**Reading rules (also in the specification):** results are conditional on the fitted folds and the 200 explained children; "reduced" requires effect ≥ 0.05, a Bonferroni-adjusted interval above 0 and preserved faithfulness; a null or negative result is a valid result. Nothing here is tuned on results. The protocol was developed over three pilots on the same data: describe this as *pre-specified before the M1 run*, not as pre-registered.

**Before you run:** Colab → install cell → restart the runtime if asked → upload `BDKR81FL.DTA` to `/content` → run in order with `MODE = "PILOT"` first (wiring), then `"FULL"` unchanged. The notebook writes only variable-level tables; no case IDs, cluster IDs or raw rows. Do not upload the DHS file anywhere else.

In [ ]:
# Colab only. The explainer libraries are PINNED to the pilot-v2 environment (shap 0.52.0, lime 0.2.0.1, captum 0.9.0). The pilot ran on Colab Python 3.13 / numpy 2.1.3 / pandas 2.2.3 / scikit-learn 1.6.1 / torch 2.11.0+cpu;
# run_metadata.json records what actually ran, and the fold hash check in section 3 fails if scikit-learn changed the cluster-grouped split.
!pip -q install "shap==0.52.0" "lime==0.2.0.1" "captum==0.9.0"

## 1. PRE-SPECIFIED SETTINGS
Edit only before the first real run. Every value here is written to `run_metadata.json`.

In [ ]:
import os, sys, glob, json, time, math, warnings, itertools, platform, random
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

MODE = "FULL"            # "SMOKE" (2-3 min wiring test) | "PILOT" | "FULL"
FEATURE_SET = "S0"        # "S0" = 14 background variables (primary). "S5" = S0 + v438 (maternal height; exploratory, needs advisor approval)

# --- seeds (fresh split; the earlier pilot used 20261007) ---
SEED_SPLIT, SEED_INST, SEED_NOISE, SEED_BOOT, SEED_NULL = 20261016, 20261017, 20261018, 20261019, 20261020
SEED_BG = 20261021                                # SHAP background subsample: fixed per fold (SEED_BG + fold), so the E-axis floor isolates permutation randomness
WORK = {}                                         # measured seconds of scheduled work (probe, main run, controls, TreeSHAP check) -> gate G1
R_E, R_T = 5, 5                                   # explainer-seed runs, training-seed runs
ES = [1001 + i for i in range(R_E)]               # explainer seeds
TS = [2001 + i for i in range(R_T)]               # training seeds

# --- explanation budgets (identical in PILOT and FULL; only the number of explained children differs) ---
N_PER_FOLD = {"SMOKE": 4, "PILOT": 20, "FULL": 40}[MODE]
N_PERM, B_SHAP = 10, 50                           # Permutation-SHAP permutations (forward+reverse pass each) and background rows
LIME_SAMPLES   = 5000
IG_STEPS       = 100
TOPK           = 5                                # top-k for Jaccard (of 14 variables)
if MODE == "SMOKE":
    R_E, R_T, N_PERM, LIME_SAMPLES, IG_STEPS = 3, 3, 3, 500, 30
    ES, TS = ES[:R_E], TS[:R_T]

# --- fixed model settings (taken from the earlier pilot's tuning; NOT re-tuned here) ---
HP = dict(LR=dict(C=0.1), RF=dict(n_estimators=300, max_depth=5, min_samples_leaf=5),
          MLP=dict(hidden=32, epochs=90, lr=1e-3, wd=1e-4, batch=128))
N_OUTER_FOLDS = 5

# --- gate thresholds (proposed; edit BEFORE running, never after seeing results) ---
GATES = dict(
    max_projected_hours=4.0,          # G1 compute: projected total run time
    min_auc_weighted=0.58,            # G2 models are better than trivial (survey-weighted AUROC, mean over folds)
    max_auc_gap_vs_earlier=0.04,      # G2b reproduces earlier S0 AUROCs (LR .624, RF .644, MLP .621) within this gap
    null_spearman_abs=0.05,           # G3 random-ranking null Spearman must be ~0
    max_additivity_err=1e-3,          # G4 SHAP local accuracy
    max_ig_delta=1e-2,                # G4 IG completeness
    noise_min_median_rank=11,         # G5 negative control: noise column median rank must be >= 11 of 15 (1 = most important) ...
    noise_max_share_ratio=0.5,        # ... and its mean importance share must be <= 0.5 x the uniform share (1/15)
    max_kappa_ci_width=0.30,          # G6 precision: 95% CI width of kappa (Spearman, SHAP vs LIME) per model
    min_floor_minus_null=0.05,        # G7 floor must exceed the null by this much, else kappa is unstable
    max_tie_share=0.01,               # G8 share of attribution vectors that are all-zero or tied at the top-k boundary
)
EARLIER_S0_AUC = dict(LR=0.624, RF=0.644, MLP=0.621)

# --- variables (order = earlier manifest) ---
S0_VARS = ["b4", "b19", "bord", "v012", "v106", "v190", "v025", "v024", "v130", "v136", "v151", "v113", "v116", "v212"]
CAT_VARS_ALL = ["b4", "v025", "v024", "v130", "v151", "v113", "v116"]
VARS = S0_VARS + (["v438"] if FEATURE_SET == "S5" else [])
CAT_VARS = [v for v in VARS if v in CAT_VARS_ALL]
SPECIAL_CAT_MISSING = {"v113": [97], "v116": [97]}       # 97 = not a de-jure resident -> structural-missing level (-1). v116 code 15 ("flush, don't know where") is a real answer and is KEPT as its own level (the earlier pilot merged it into missing; deviation recorded)
DATA_PATH = os.environ.get("THESIS_DATA_PATH")           # leave None: auto-search /content
OUT_DIR = "/content/outputs_m1" if os.path.isdir("/content") else "./outputs_m1"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODE={MODE}  FEATURE_SET={FEATURE_SET}  explained children/fold={N_PER_FOLD}  runs: R_E={R_E}, R_T={R_T}")

## 2. Libraries, reproducibility, data

In [ ]:
import torch, torch.nn as nn
import shap, lime, lime.lime_tabular
from captum.attr import IntegratedGradients
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from importlib import metadata as _md0
def _v0(p):
    try: return _md0.version(p)
    except Exception: return 'n/a'
PINNED = dict(shap='0.52.0', lime='0.2.0.1', captum='0.9.0')
_drift = {p: (v, _v0(p)) for p, v in PINNED.items() if _v0(p) != v}
ALLOW_VERSION_DRIFT = os.environ.get('THESIS_ALLOW_VERSION_DRIFT') == '1'          # relaxes ONLY the explainer-library pin check
ALLOW_HASH_MISMATCH = os.environ.get('THESIS_ALLOW_HASH_MISMATCH') == '1'          # relaxes ONLY the fold-hash check (needed if scikit-learn differs); the outcome hash stays enforced
if _drift and not ALLOW_VERSION_DRIFT:
    raise RuntimeError(f'Explainer library versions differ from the pre-specified pins (pinned, found): {_drift}. Re-run the install cell and restart the runtime.')
print('explainer libraries:', {p: _v0(p) for p in PINNED}, '| sklearn', sklearn.__version__, '| numpy', np.__version__, '| torch', torch.__version__)

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_all_seeds(SEED_SPLIT)
torch.set_num_threads(1)      # single thread: float summation order (hence trained MLP weights, IG) must not depend on the host's core count

In [ ]:
def find_dta():
    if DATA_PATH and os.path.exists(DATA_PATH): return DATA_PATH
    pats = ["/content/*.[dD][tT][aA]", "/content/*/*.[dD][tT][aA]", "./*.[dD][tT][aA]"]     # Drive is searched only if you set DATA_PATH
    hits = [p for pat in pats for p in glob.glob(pat) if "/drive/" not in p]
    pref = [p for p in hits if os.path.basename(p).upper().startswith("BDKR81FL")]
    if pref: return pref[0]
    if hits: return hits[0]
    raise FileNotFoundError("Upload BDKR81FL.DTA to /content (Colab file panel) or set DATA_PATH, then re-run this cell.")

path = find_dta()
need = ["b5", "hw70", "v001", "v005"] + VARS
try:
    raw = pd.read_stata(path, columns=need, convert_categoricals=False)
except (KeyError, ValueError):
    raw = pd.read_stata(path, convert_categoricals=False)
    raw.columns = [c.lower() for c in raw.columns]
    raw = raw[need]
print("loaded", os.path.basename(path), raw.shape)

d = raw[(raw["b5"] == 1) & (raw["hw70"] < 9990)].copy()          # living children with a valid HAZ
y = (d["hw70"] < -200).astype(int).values                         # stunted: HAZ < -2
w = (d["v005"] / 1e6).values                                      # DHS sample weight
groups = d["v001"].astype(int).values                             # cluster (PSU)
Xdf = d[VARS].astype(float).copy()
_BAD = {9, 99, 999, 9999}                                         # undocumented DHS 'missing/NIU' style codes: none occurred in the BDHS 2022 pilot cohort (checked), so any appearance means a changed file or filter
for v in CAT_VARS:
    _hit = sorted(set(Xdf[v].dropna().unique()) & _BAD)
    assert not _hit, f'categorical predictor {v} contains undocumented special code(s) {_hit}: map them explicitly before running'
for v in VARS:
    if v not in CAT_VARS and v != 'v438': assert Xdf[v].max() < 90, f'numeric predictor {v} has max {Xdf[v].max()} (special code?)'
for v, codes in SPECIAL_CAT_MISSING.items():
    if v in Xdf: Xdf.loc[Xdf[v].isin(codes), v] = np.nan
if "v438" in Xdf: Xdf.loc[Xdf["v438"] >= 9990, "v438"] = np.nan
for v in CAT_VARS: Xdf[v] = Xdf[v].fillna(-1.0)                  # -1 = structural / missing level
X = Xdf.values
import hashlib
X_SHA = hashlib.sha256(np.ascontiguousarray(X, dtype=np.float64).tobytes()).hexdigest()   # digest of the encoded predictor matrix (features in VARS order, special codes handled); recorded, to be pinned after the first FULL run
print('predictor matrix sha256:', X_SHA[:16])
EXPECTED_X_SHA256 = "3eb260a03428d8e08e3b2d34bd6d9be932576a71445c9202b307dda3e544093a"   # recorded in the v3.1 PILOT run on BDKR81FL.DTA (the predictor matrix does not depend on MODE)
if len(y) == 4105 and int(y.sum()) == 966 and os.environ.get('THESIS_ALLOW_HASH_MISMATCH') != '1':
    assert X_SHA == EXPECTED_X_SHA256, 'predictor matrix differs from the v3.1 pilot (data file, variable coding or special-code handling changed)'
print(f"analytic n={len(y)} (earlier pilot: 4105)  stunted={y.sum()} ({y.mean():.3%}; earlier 23.53%)  clusters={len(np.unique(groups))}")
STRICT = os.environ.get("THESIS_STRICT", "1") == "1"
if STRICT and (len(y) != 4105 or int(y.sum()) != 966):
    raise RuntimeError("Cohort differs from the earlier pilot (n=4105, stunted=966). Check the DTA file / filters, or set the environment variable THESIS_STRICT=0 deliberately.")
print("missing share per variable (raw NaN):", {v: round(float(np.isnan(Xdf[v]).mean()), 3) for v in VARS if np.isnan(Xdf[v]).any()} or "none")

## 3. Cluster-grouped, prevalence-stratified outer folds (fresh split)

In [ ]:
sgk = StratifiedGroupKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED_SPLIT)
FOLDS = list(sgk.split(X, y, groups))
for k, (tr, te) in enumerate(FOLDS):
    assert len(set(groups[tr]) & set(groups[te])) == 0, "cluster leakage across train/test"
import hashlib
fold_id = np.zeros(len(y), dtype=np.int8)
for k, (tr, te) in enumerate(FOLDS): fold_id[te] = k
FOLD_SHA = hashlib.sha256(fold_id.tobytes()).hexdigest(); Y_SHA = hashlib.sha256(y.astype(np.int8).tobytes()).hexdigest()
print("fold assignment sha256:", FOLD_SHA[:16], "| outcome vector sha256:", Y_SHA[:16], "| sum of weights:", round(float(w.sum()), 3), "| sklearn", sklearn.__version__)
EXPECTED_FOLD_SHA256 = "8a4f5168872e387ede8307484daf8b360c76abfc3731165f9216ab84ab925667"   # pilot v2 (sklearn 1.6.1)
EXPECTED_Y_SHA256 = "ee4c36ab6cfd2a48ba2a5ee4f9139134fac8ee4e237e582321a0ba0e4c84c410"
if len(y) == 4105 and int(y.sum()) == 966:
    assert Y_SHA == EXPECTED_Y_SHA256, "outcome vector differs from the pilot (data file or filters changed)"
    assert ALLOW_HASH_MISMATCH or FOLD_SHA == EXPECTED_FOLD_SHA256, "fold assignment differs from the pilot: scikit-learn version drift in StratifiedGroupKFold? Re-pin scikit-learn==1.6.1 (or set THESIS_ALLOW_HASH_MISMATCH=1 deliberately; the override is recorded in run_metadata.json)."
print(pd.DataFrame([dict(fold=k, n_train=len(tr), n_test=len(te), test_clusters=len(set(groups[te])),
                         test_prev=round(y[te].mean(), 3)) for k, (tr, te) in enumerate(FOLDS)]).to_string(index=False))

## 4. Preprocessing (fit on the training fold only), models
* Numeric variables: median imputation → standardisation. Categorical variables: one-hot (unknown level → all zeros).
* Explanations are computed at the **source-variable level** (14 players): SHAP and LIME perturb whole variables; IG runs on the encoded inputs and is summed over each variable's dummy columns (valid for IG because the straight-line path moves all dummies of a variable together).
* Explained quantity for every method: predicted probability of the positive class (class-balanced training, so the score is not calibrated to the 23.5% prevalence).

In [ ]:
class Prep:
    def __init__(self, vars_, cat_vars):
        self.vars, self.cat = list(vars_), [v in cat_vars for v in vars_]
    def impute(self, X):
        X = np.array(X, dtype=float, copy=True)
        for j in range(X.shape[1]):
            m = np.isnan(X[:, j])
            if m.any(): X[m, j] = -1.0 if self.cat[j] else self.med[j]
        return X
    def fit(self, X):
        X = np.asarray(X, float)
        for j, c in enumerate(self.cat):
            if not c and np.isnan(X[:, j]).all(): raise ValueError(f'numeric predictor {self.vars[j]} is entirely missing in this training fold')
        self.med = {j: float(np.nanmedian(X[:, j])) for j, c in enumerate(self.cat) if not c}
        Xi = self.impute(X)
        self.mu = {j: Xi[:, j].mean() for j, c in enumerate(self.cat) if not c}
        self.sd = {j: (Xi[:, j].std() or 1.0) for j, c in enumerate(self.cat) if not c}
        self.levels = {j: np.unique(Xi[:, j]) for j, c in enumerate(self.cat) if c}
        self.zcols, self.zvar = [], []
        for j, v in enumerate(self.vars):
            if self.cat[j]:
                for lv in self.levels[j]: self.zcols.append(f"{v}={lv:g}"); self.zvar.append(j)
            else:
                self.zcols.append(v); self.zvar.append(j)
        self.zvar = np.array(self.zvar)
        self.G = np.zeros((len(self.zvar), len(self.vars)))
        self.G[np.arange(len(self.zvar)), self.zvar] = 1.0           # encoded column -> source variable
        return self
    def encode(self, Xi):
        Xi = np.asarray(Xi, float); parts = []
        for j in range(Xi.shape[1]):
            if self.cat[j]: parts.append((Xi[:, [j]] == self.levels[j][None, :]).astype(float))
            else: parts.append(((Xi[:, [j]] - self.mu[j]) / self.sd[j]))
        return np.hstack(parts)

class MLPNet(nn.Module):
    def __init__(self, p, h):
        super().__init__(); self.f = nn.Sequential(nn.Linear(p, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, z): return self.f(z).squeeze(-1)          # logit

class ProbModule(nn.Module):                                      # probability output, shape (n,), for Captum
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, z): return torch.sigmoid(self.net(z))

class LinLogit(nn.Module):                                        # logistic regression as a float64 torch module (logit output)
    def __init__(self, coef, intercept):
        super().__init__(); self.lin = nn.Linear(coef.shape[1], 1).double()          # float64: the twin reproduces sklearn's float64 LR
        with torch.no_grad():
            self.lin.weight.copy_(torch.as_tensor(coef, dtype=torch.float64)); self.lin.bias.copy_(torch.as_tensor(intercept, dtype=torch.float64))
    def forward(self, z): return self.lin(z).squeeze(-1)

class Fitted:
    def __init__(self, kind, clf=None, net=None): self.kind, self.clf, self.net = kind, clf, net
    def proba(self, Z):
        if self.net is None: return self.clf.predict_proba(Z)[:, 1]
        out = []
        with torch.no_grad():
            for i in range(0, len(Z), 8192):
                out.append(torch.sigmoid(self.net(torch.as_tensor(Z[i:i + 8192], dtype=torch.float32))).numpy())
        return np.concatenate(out).astype(float)
    def prob_module(self):                                          # None for RF (IG not applicable)
        if self.kind == "RF": return None
        net = self.net if self.kind == "MLP" else LinLogit(self.clf.coef_, self.clf.intercept_)   # LR twin: same function as sklearn
        net = net.eval()
        for q in net.parameters(): q.requires_grad_(False)
        return ProbModule(net).eval()

def fit_model(kind, Z, yv, seed):
    if kind == "LR":
        clf = LogisticRegression(C=HP["LR"]["C"], class_weight="balanced", max_iter=5000).fit(Z, yv)
        return Fitted("LR", clf=clf)
    if kind == "RF":
        clf = RandomForestClassifier(class_weight="balanced_subsample", random_state=seed, n_jobs=-1, **HP["RF"]).fit(Z, yv)
        clf.set_params(n_jobs=1)                                    # fit in parallel (tree seeds are independent), PREDICT serially: parallel predict_proba sums trees in a nondeterministic order
        return Fitted("RF", clf=clf)
    h = HP["MLP"]; torch.manual_seed(seed)
    net = MLPNet(Z.shape[1], h["hidden"]); opt = torch.optim.Adam(net.parameters(), lr=h["lr"], weight_decay=h["wd"])
    pw = torch.tensor((1 - yv).sum() / max(yv.sum(), 1), dtype=torch.float32)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    Zt, yt = torch.as_tensor(Z, dtype=torch.float32), torch.as_tensor(yv, dtype=torch.float32)
    gen = torch.Generator().manual_seed(seed)                       # created ONCE: a fresh shuffle every epoch, reproducible from the seed
    for _ in range(h["epochs"]):
        perm = torch.randperm(len(Zt), generator=gen)
        for i in range(0, len(Zt), h["batch"]):
            b = perm[i:i + h["batch"]]; opt.zero_grad(); lossf(net(Zt[b]), yt[b]).backward(); opt.step()
    net.eval()
    return Fitted("MLP", net=net)

def make_predict_raw(prep, fm):
    return lambda Xi: fm.proba(prep.encode(np.asarray(Xi, float)))

## 5. Explainers
*Explainer seed* = the permutation order (SHAP) / the perturbation stream (LIME). The SHAP background (50 training rows) is drawn **once per fold** (`SEED_BG + fold`), so the E-axis floor isolates the explainer's own randomness. `SHAP_bg` is a separate *sensitivity axis*: permutation seed fixed, only the background draw varies (reported as its own floor, never mixed into the primary one). IG is deterministic given the model, so its explainer-seed floor is exactly 1 and only retraining variability exists.

In [ ]:
METHODS_BY_MODEL = dict(LR=["SHAP", "LIME", "IG"], RF=["SHAP", "LIME"], MLP=["SHAP", "LIME", "IG"])

SHAP_PERM_LOG = []                                                  # effective permutations per call (minimum over the explained children)
def shap_perm(f, Xtr_imp, Xin, seed, bg_seed):
    p = Xin.shape[1]
    rng = np.random.RandomState(bg_seed)
    bg = Xtr_imp[rng.choice(len(Xtr_imp), min(B_SHAP, len(Xtr_imp)), replace=False)]
    expl = shap.PermutationExplainer(f, shap.maskers.Independent(bg, max_samples=len(bg)), seed=seed)
    me = N_PERM * (2 * p + 1)
    ex = expl(Xin, max_evals=me, silent=True)                         # shap uses npermutations = max_evals // (2*n_varying + 1); n_varying = variables that differ from the background somewhere
    nvary = (Xin[:, None, :] != bg[None, :, :]).any(1).sum(1)
    SHAP_PERM_LOG.append(int((me // (2 * nvary + 1)).min()))          # == N_PERM unless a variable is constant against the whole background
    vals = np.asarray(ex.values, float)
    add_err = np.abs(vals.sum(1) + np.asarray(ex.base_values, float).reshape(-1) - f(Xin))
    return vals, add_err

def lime_run(f, Xtr_imp, Xin, cat_idx, names, seed):
    expl = lime.lime_tabular.LimeTabularExplainer(
        Xtr_imp, feature_names=names, categorical_features=cat_idx, mode="classification",
        discretize_continuous=True, feature_selection="none", random_state=seed)
    def pf(a):
        p1 = f(a); return np.column_stack([1 - p1, p1])
    out = np.zeros(Xin.shape); r2 = np.zeros(len(Xin))
    for i in range(len(Xin)):
        e = expl.explain_instance(Xin[i], pf, labels=(1,), num_features=Xin.shape[1], num_samples=LIME_SAMPLES)
        for j, wgt in e.local_exp[1]: out[i, j] = wgt
        r2[i] = e.score[1] if isinstance(e.score, dict) else e.score
    return out, r2

def ig_run(fm, prep, Zin, Ztr, Xtr_imp, variants=True):
    pm = fm.prob_module()
    ig = IntegratedGradients(pm)
    dt = torch.float64 if fm.kind == 'LR' else torch.float32          # LR twin is float64; the MLP is a float32 network
    zin = torch.as_tensor(Zin, dtype=dt)
    res, deltas = {}, []
    # baseline A (primary): training mean of the encoded inputs
    b_mean = torch.as_tensor(Ztr.mean(0, keepdims=True), dtype=dt).expand_as(zin)
    a, dl = ig.attribute(zin, baselines=b_mean, n_steps=IG_STEPS, method="gausslegendre", return_convergence_delta=True)
    a = a.numpy().astype(float); deltas.append(np.abs(dl.numpy()))
    res["IG"] = a @ prep.G                                           # signed sum over a variable's dummy columns
    if variants:
        res["IG_sumabs"] = np.abs(a) @ prep.G                       # aggregation sensitivity: sum of |dummy attributions|
        # baseline B (sensitivity): a realisable "typical child" = median numeric / modal category of the training fold
        base_raw = np.array([prep.levels[j][np.bincount(np.searchsorted(prep.levels[j], Xtr_imp[:, j])).argmax()]
                             if prep.cat[j] else prep.med[j] for j in range(Xtr_imp.shape[1])])[None, :]
        b_med = torch.as_tensor(prep.encode(base_raw), dtype=dt).expand_as(zin)
        a2, dl2 = ig.attribute(zin, baselines=b_med, n_steps=IG_STEPS, method="gausslegendre", return_convergence_delta=True)
        res["IG_medmode"] = a2.numpy().astype(float) @ prep.G; deltas.append(np.abs(dl2.numpy()))
    return res, deltas                                                # [primary |delta| array, medmode |delta| array (if variants)]

In [ ]:
def pick_instances(te, groups, n, seed):
    rng = np.random.RandomState(seed); by = {}
    for i in te: by.setdefault(groups[i], []).append(i)
    ids = sorted(by); rng.shuffle(ids)
    return np.array([by[c][rng.randint(len(by[c]))] for c in ids[:n]])

## 6a. Agreement metrics (defined here so they can be unit-tested; used in section 9)

In [ ]:
def _top(v, k): return set(np.argsort(-np.abs(v), kind="stable")[:k])
def sim_tv(a, b):
    pa, pb = np.abs(a), np.abs(b); sa, sb = pa.sum(), pb.sum()
    pa = pa / sa if sa > 0 else np.full_like(pa, 1 / len(pa)); pb = pb / sb if sb > 0 else np.full_like(pb, 1 / len(pb))
    return 1.0 - 0.5 * np.abs(pa - pb).sum()
def sim_spearman(a, b):
    ra, rb = rankdata(np.abs(a)), rankdata(np.abs(b))
    if ra.std() == 0 or rb.std() == 0: return 0.0
    return float(np.corrcoef(ra, rb)[0, 1])
def sim_jaccard(a, b, k=TOPK):
    A, B = _top(a, k), _top(b, k); return len(A & B) / len(A | B)
def sim_rbo(a, b, p=0.9):
    oa, ob = np.argsort(-np.abs(a), kind="stable"), np.argsort(-np.abs(b), kind="stable"); D = len(a)
    wts = p ** np.arange(D); ag = [len(set(oa[:d]) & set(ob[:d])) / d for d in range(1, D + 1)]
    return float((wts * np.array(ag)).sum() / wts.sum())
METRICS = {"1-TV": sim_tv, "Spearman": sim_spearman, f"Jaccard@{TOPK}": sim_jaccard, "RBO": sim_rbo}

EPS = 1e-12
def kappa_parts(mC, mA, mB, mN):
    """(ceiling, kappa, excess disagreement). kappa is NaN (undefined, never inf) when 1-N ~ 0 or when the ceiling is at the null (a floor is at/below the null)."""
    mC, mA, mB, mN = (np.asarray(v, float) for v in (mC, mA, mB, mN))
    with np.errstate(divide="ignore", invalid="ignore"):
        d1 = 1 - mN
        rA = np.where(np.abs(d1) > EPS, np.clip((mA - mN) / d1, 0, 1), np.nan)
        rB = np.where(np.abs(d1) > EPS, np.clip((mB - mN) / d1, 0, 1), np.nan)
        ceil = mN + d1 * np.sqrt(rA * rB)
        d2 = ceil - mN
        kap = np.where(np.abs(d2) > EPS, (mC - mN) / d2, np.nan)
    return ceil, kap, ceil - mC
def cons_floor(FE, FT):
    """Conservative per-child floor = min(E-axis floor, T-axis floor) child by child where a T-axis floor exists; E-axis only otherwise (LR)."""
    FE, FT = np.asarray(FE, float), np.asarray(FT, float)
    return np.where(np.isnan(FT), FE, np.minimum(FE, FT))
def cons_floor3(FE, FT, FB):
    """Conservative per-child floor = min over the explainer-seed (E), retrain (T) and SHAP-background (B) axes, child by child; any axis that does not exist is NaN and ignored."""
    parts = np.vstack([np.asarray(v, float) for v in (FE, FT, FB)])
    return np.nanmin(parts, axis=0)
def kappa_status(mN, kap):
    return "ok" if np.isfinite(kap) else ("UNDEFINED: null ~ 1" if abs(1 - mN) <= EPS else "UNDEFINED: a floor is at/below the null")

In [ ]:
def sim_vec(A, B, fn): return np.array([fn(A[i], B[i]) for i in range(len(A))])
def mean_pairs(runsA, runsB, fn, same):
    vals = []
    for r, s in itertools.product(range(len(runsA)), range(len(runsB))):
        if same and s <= r: continue
        vals.append(sim_vec(runsA[r], runsB[s], fn))
    return np.mean(vals, axis=0) if vals else None
def n2_child(A0, B0, fn):                                           # A(child i) vs B(every other child j) within the fold, symmetrised over both directions
    n = len(A0)
    if n < 2: return np.full(n, np.nan)
    return np.array([0.5 * np.mean([fn(A0[i], B0[j]) + fn(A0[j], B0[i]) for j in range(n) if j != i]) for i in range(n)])

## M1.0 Module settings (specification v2, frozen before the first M1 run)
* Arms: **M0** default configuration (recomputed here and checked against the FULL run), **M1** one shared 50-row reference set for SHAP, LIME and IG, **C1** control: independent 50-row reference sets per explainer.
* `R_B` reference draws × `R_EM` explainer seeds per arm. Reference draw 0 is the draw the main run used for the SHAP background (`SEED_BG + fold`).
* Primary outcome: change in child-specific agreement Δ = (C − N2)[M1] − (C − N2)[M0], Spearman, 7 model×pair cells. Decision rule and faithfulness criteria are in the specification and in section M1.7.

In [ ]:
R_B, R_EM = 5, 3                     # reference draws, explainer seeds per draw
K_REF = 50                           # rows in a reference set (equals B_SHAP so that M1 draw 0 == the main-run SHAP background)
K_NN, N_MARG, N_RAND, DEL_K = 50, 100, 30, 7     # conditional-deletion neighbours, marginal-deletion rows, random orders, largest deleted set
N_REAL_CHILD, N_REAL_ROWS = 8, 20    # realism diagnostics: children per fold, composite rows per set
IG_STEPS_B = 2 * IG_STEPS            # steps per baseline for the baseline-averaged IG (M1, C1): real-child baselines can be far from x, so more steps keep per-baseline completeness well inside the gate; M0 keeps IG_STEPS to reproduce the main run
DELTA_MIN, FAITH_MARGIN = 0.05, 0.10 # practical-size threshold for Delta; non-inferiority margin for faithfulness
M1_GATES = dict(max_projected_hours=3.0, repro_tol_C=0.02 if MODE == "FULL" else 0.04, repro_tol_floor=0.03 if MODE == "FULL" else 0.05,
                max_additivity_err=1e-3, max_ig_delta=1e-2, max_lime_r2_drop=0.10, max_constraint_baseline_viol=0.02)
if MODE == "SMOKE":
    R_B, R_EM, N_RAND, N_MARG, K_NN, N_REAL_CHILD, N_REAL_ROWS = 2, 2, 5, 20, 20, 2, 5
assert R_EM <= len(ES) and K_REF == B_SHAP, "R_EM must not exceed the number of explainer seeds; K_REF must equal B_SHAP (draw 0 == main-run background)"
MODELS = ["LR", "RF", "MLP"]
CELLS = [(m, a, b) for m in MODELS for a, b in itertools.combinations(METHODS_BY_MODEL[m], 2)]      # 7 model x pair cells
N_CELLS = len(CELLS); assert N_CELLS == 7
# FULL-run reference values (cross_method_vs_floor.csv / noise_floor.csv of the v3.1.1 FULL run, Spearman) used for the reproduction gate U4
FULL_REF_C = {("LR", "SHAP", "LIME"): 0.7303534, ("LR", "SHAP", "IG"): 0.9296264, ("LR", "LIME", "IG"): 0.7740176, ("RF", "SHAP", "LIME"): 0.6754031,
              ("MLP", "SHAP", "LIME"): 0.5026734, ("MLP", "SHAP", "IG"): 0.6817978, ("MLP", "LIME", "IG"): 0.3799868}
FULL_REF_FLOOR_E = {("LR", "SHAP"): 0.9992264, ("LR", "LIME"): 0.9423033, ("RF", "SHAP"): 0.9921451, ("RF", "LIME"): 0.8969604,
                    ("MLP", "SHAP"): 0.9756923, ("MLP", "LIME"): 0.9064484}
print(f"M1 settings: R_B={R_B} R_EM={R_EM} K_REF={K_REF} | deletion: K_NN={K_NN} N_MARG={N_MARG} N_RAND={N_RAND} DEL_K={DEL_K} | cells={N_CELLS}")

## M1.1 Reference sets, explainers on explicit references
* `make_refs`: three independent 50-row draws from the training fold (rows are real, imputed training children): B (shared, M1), B' (LIME in C1), B'' (IG in C1). Draw `r = 0` of B uses the main-run seed `SEED_BG + fold`.
* `shap_with_bg`: Permutation SHAP with an explicit background (same code path as `shap_perm`).
* `ig_run_B`: IG averaged over every baseline in the reference set. Signed attributions are averaged first, then summed over each variable's dummy columns (the same pipeline as the primary IG); |·| is taken later, as for every other explainer.
* LIME on a reference set: `lime_run(f, B, ...)`, i.e. the reference set is LIME's `training_data` (bin edges, category frequencies, scales).

In [ ]:
def make_refs(k, n_train, r):
    base = SEED_BG + (0 if r == 0 else 1000 * r) + k
    draw = lambda seed: np.random.RandomState(seed).choice(n_train, min(K_REF, n_train), replace=False)
    return draw(base), draw(base + 100), draw(base + 200)             # B (shared), B' (independent, LIME in C1), B'' (independent, IG in C1)

def shap_with_bg(f, bg, Xin, seed):
    p = Xin.shape[1]
    expl = shap.PermutationExplainer(f, shap.maskers.Independent(bg, max_samples=len(bg)), seed=seed)
    me = N_PERM * (2 * p + 1)
    ex = expl(Xin, max_evals=me, silent=True)
    vals = np.asarray(ex.values, float)
    add_err = np.abs(vals.sum(1) + np.asarray(ex.base_values, float).reshape(-1) - f(Xin))
    return vals, add_err

def ig_run_B(fm, prep, Zin, ZB, module=None):
    """IG averaged over every baseline row. Returns (variable-level attribution, max per-baseline |completeness delta|, max |completeness error| of the AVERAGED attribution)."""
    pm = module if module is not None else fm.prob_module()
    ig = IntegratedGradients(pm)
    dt = torch.float64 if fm.kind == "LR" else torch.float32
    zin = torch.as_tensor(Zin, dtype=dt); acc = np.zeros(Zin.shape); dmax = 0.0
    for b in ZB:
        base = torch.as_tensor(np.asarray(b)[None, :], dtype=dt).expand_as(zin)
        a, dl = ig.attribute(zin, baselines=base, n_steps=IG_STEPS_B, method="gausslegendre", return_convergence_delta=True)
        acc += a.numpy().astype(float); dmax = max(dmax, float(np.abs(dl.numpy()).max()))
    with torch.no_grad():
        target = (pm(zin) - pm(torch.as_tensor(np.asarray(ZB), dtype=dt)).mean()).numpy().astype(float)           # f(x) - mean over baselines of f(b)
    avg_err = float(np.abs((acc / len(ZB)).sum(1) - target).max())
    return (acc / len(ZB)) @ prep.G, dmax, avg_err

## M1.2 Realistic deletion-faithfulness (specification section 4)
Deleting a set S from child x = replacing x's values of S by those of real children. **Conditional** (primary): the `K_NN` training children closest to x on the *kept* variables (Gower distance), S copied jointly from one real child. **Marginal** (secondary): S copied from each of `N_MARG` real training children (the deletion reference set, disjoint from every reference set). Score = mean over k = 1…7 of |mean f(composites) − f(x)|; random baseline = the same score averaged over `N_RAND` random orders.

In [ ]:
def make_scheme_data(Xtr, prep, Rm):
    lo, hi = np.percentile(Xtr, 5, axis=0), np.percentile(Xtr, 95, axis=0); rng_ = hi - lo; rng_[rng_ <= 0] = 1.0
    return dict(Xtr=Xtr, cat=np.array(prep.cat), rng=rng_, R=Rm)

def gower_matrix(SC, x):                                              # (n_vars, n_train): distance of x to every training child, per variable
    Xtr = SC["Xtr"]; D = np.empty((Xtr.shape[1], len(Xtr)))
    for j in range(Xtr.shape[1]):
        D[j] = (Xtr[:, j] != x[j]).astype(float) if SC["cat"][j] else np.minimum(np.abs(Xtr[:, j] - x[j]) / SC["rng"][j], 1.0)
    return D

def comps_marg(x, S, SC):
    Xc = np.repeat(x[None, :], len(SC["R"]), 0); Xc[:, list(S)] = SC["R"][:, list(S)]; return Xc

def comps_cond(x, S, SC, Dm, cache):
    key = tuple(sorted(S))
    if key not in cache:
        kept = [j for j in range(Dm.shape[0]) if j not in key]
        dist = Dm[kept].mean(0); nn = np.lexsort((np.arange(len(dist)), dist))[:K_NN]       # stable, deterministic
        Xc = np.repeat(x[None, :], len(nn), 0); Xc[:, list(key)] = SC["Xtr"][nn][:, list(key)]
        cache[key] = Xc
    return cache[key]

def fbatch(f, X, chunk=100000):
    return np.concatenate([f(X[i:i + chunk]) for i in range(0, len(X), chunk)])

def faith_child(f, x, f0, Dm, SC, orders, rand_orders):
    """Deletion scores of each ordering and the random-order mean, for the conditional and the marginal scheme."""
    allord = [np.asarray(o) for o in list(orders) + list(rand_orders)]; out = {}
    cache = {}
    for scheme in ("cond", "marg"):
        blocks = []
        for o in allord:
            for kk in range(1, DEL_K + 1):
                S = o[:kk]
                blocks.append(comps_cond(x, S, SC, Dm, cache) if scheme == "cond" else comps_marg(x, S, SC))
        sizes = np.array([len(b) for b in blocks]); P = fbatch(f, np.vstack(blocks))
        means = np.add.reduceat(P, np.concatenate([[0], np.cumsum(sizes)[:-1]])) / sizes
        D = np.abs(means - f0).reshape(len(allord), DEL_K).mean(1)
        out[scheme] = (D[:len(orders)], float(D[len(orders):].mean()))
    return out

def constraint_viol(X):
    """Logical constraints between mother's age and the child's age (rows of imputed raw predictors); returns (C1, C2) boolean violation vectors."""
    ib, i12, ibo, i212 = VARS.index("b19"), VARS.index("v012"), VARS.index("bord"), VARS.index("v212")
    age_at_birth = X[:, i12] - X[:, ib] / 12.0
    c1 = X[:, i212] > age_at_birth + 1.0
    c2 = (X[:, ibo] == 1) & (np.abs(X[:, i212] - age_at_birth) > 2.0)
    return c1, c2

def nn_gower(SC, C):                                                   # distance from each composite row to its nearest real training child
    Xtr = SC["Xtr"]; out = []
    for c in C:
        d = np.zeros(len(Xtr))
        for j in range(Xtr.shape[1]):
            d += ((Xtr[:, j] != c[j]).astype(float) if SC["cat"][j] else np.minimum(np.abs(Xtr[:, j] - c[j]) / SC["rng"][j], 1.0))
        out.append((d / Xtr.shape[1]).min())
    return np.array(out)

def realism_diag(x, Dm, SC, seed):
    rng = np.random.RandomState(seed); rows = []; cache = {}
    real_nn = float(np.min(Dm.mean(0)))
    for kk in (3, 7):
        S = rng.permutation(Dm.shape[0])[:kk]
        for scheme in ("cond", "marg"):
            Xc = comps_cond(x, S, SC, Dm, cache) if scheme == "cond" else comps_marg(x, S, SC)
            Xc = Xc[:N_REAL_ROWS]; c1, c2 = constraint_viol(Xc)
            rows.append(dict(scheme=scheme, k=kk, viol_C1=float(c1.mean()), viol_C2=float(c2.mean()), median_nn_gower=float(np.median(nn_gower(SC, Xc))), real_child_nn_gower=real_nn))
    return rows

## M1.3 Stage gate 0 — unit tests (seconds to a minute)
Metric sanity (inherited), reference-draw identity, **U1** LR logit identity (IG averaged over the reference set == linear SHAP), **U2** LIME on a 50-row reference, scoring-code check **U5**. A failure here is a code problem, not a result.

In [ ]:
UNIT_TESTS_PASSED = False; _t_unit = time.time(); UT = {}
_a = np.array([5., 4, 3, 2, 1, 0.5, 0.4, 0.3, 0.2, 0.1, 0.05, 0.04, 0.03, 0.02]); _b = _a[::-1].copy()
assert abs(sim_tv(_a, _a) - 1) < 1e-12 and abs(sim_spearman(_a, _a) - 1) < 1e-12 and sim_jaccard(_a, _a) == 1 and abs(sim_rbo(_a, _a) - 1) < 1e-12
assert abs(sim_spearman(_a, _b) + 1) < 1e-12 and sim_jaccard(_a, _b) == 0 and abs(sim_tv(-_a, _a) - 1) < 1e-12
assert abs(sim_rbo(np.array([3., 2, 1]), np.array([2., 3, 1]), p=0.5) - 3 / 7) < 1e-12, "RBO known-value test"
_tr, _te = FOLDS[0]; _pr = Prep(VARS, CAT_VARS).fit(X[_tr]); _Xtr = _pr.impute(X[_tr]); _Ztr = _pr.encode(_Xtr)
_sel = pick_instances(_te, groups, 4, SEED_INST); assert len(set(groups[_sel])) == len(_sel)
_Xin = _pr.impute(X[_sel]); _Zin = _pr.encode(_Xin)
_catidx = [j for j, v in enumerate(VARS) if v in CAT_VARS]
# reference draws: draw 0 of B equals the main-run SHAP background draw; B, B', B'' differ; draws differ across r
_b0 = make_refs(0, len(_Xtr), 0); _bg_main = np.random.RandomState(SEED_BG + 0).choice(len(_Xtr), min(B_SHAP, len(_Xtr)), replace=False)
assert np.array_equal(_b0[0], _bg_main), "M1 draw 0 must equal the main-run SHAP background"
assert not np.array_equal(_b0[0], _b0[1]) and not np.array_equal(_b0[0], _b0[2]) and not np.array_equal(_b0[1], _b0[2]) and not np.array_equal(_b0[0], make_refs(0, len(_Xtr), 1)[0])
# SHAP with explicit background == main-run SHAP at the same background and seed
_fm = fit_model("LR", _Ztr, y[_tr], TS[0]); _f = make_predict_raw(_pr, _fm)
_s_main, _ = shap_perm(_f, _Xtr, _Xin, ES[0], SEED_BG + 0); _s_new, _ = shap_with_bg(_f, _Xtr[_b0[0]], _Xin, ES[0])
assert np.allclose(_s_main, _s_new), "shap_with_bg must reproduce shap_perm for the same background and seed"
# U1: LR on the LOGIT output: IG averaged over B == variable-level SHAP == w (x - mean_B) summed per variable
_B = _Xtr[_b0[0]]; _ZB = _pr.encode(_B)
_logit_mod = LinLogit(_fm.clf.coef_, _fm.clf.intercept_).eval()
_ig_logit, _, _ = ig_run_B(_fm, _pr, _Zin, _ZB, module=_logit_mod)
_closed = ((_Zin - _ZB.mean(0)) * _fm.clf.coef_.reshape(1, -1)) @ _pr.G
_f_logit = lambda Xi: _fm.clf.decision_function(_pr.encode(np.asarray(Xi, float)))
_shap_logit, _ = shap_with_bg(_f_logit, _B, _Xin, ES[0])
assert np.abs(_ig_logit - _closed).max() < 1e-8, "U1: averaged IG must equal the closed form for a linear logit"
assert np.abs(_shap_logit - _closed).max() < 1e-6, "U1: variable-level SHAP of a linear logit must equal the closed form"
UT["U1_max_abs_diff_IGavg_vs_closed"] = float(np.abs(_ig_logit - _closed).max()); UT["U1_max_abs_diff_SHAP_vs_closed"] = float(np.abs(_shap_logit - _closed).max())
# U2: LIME on a 50-row reference constructs and explains for every fold and draw (cheap settings)
_lime_ok = 0; _lime_fail = []
for _k, (_trk, _tek) in enumerate(FOLDS):
    _prk = Prep(VARS, CAT_VARS).fit(X[_trk]); _Xk = _prk.impute(X[_trk]); _fk = make_predict_raw(_prk, fit_model("LR", _prk.encode(_Xk), y[_trk], TS[0]))
    _x1 = _prk.impute(X[pick_instances(_tek, groups, 2, SEED_INST + _k)])
    for _r in range(R_B):
        try:
            for _bi in make_refs(_k, len(_Xk), _r):
                _l, _ = lime_run(_fk, _Xk[_bi], _x1, _catidx, VARS, ES[0]); assert np.isfinite(_l).all()
            _lime_ok += 1
        except Exception as _e:
            _lime_fail.append((_k, _r, type(_e).__name__, str(_e)[:80]))
assert not _lime_fail, f"U2: LIME failed on a reference set {_lime_fail[:3]} -> raise K_REF for ALL explainers together (specification) and re-run"
UT["U2_lime_reference_sets_ok"] = _lime_ok
# U5: scoring code on a toy model that depends on one variable only (child age b19)
_j = VARS.index("b19"); _ft = lambda Xi: 1 / (1 + np.exp(-(np.asarray(Xi, float)[:, _j] - _Xtr[:, _j].mean()) / _Xtr[:, _j].std()))
_Rm = _Xtr[np.random.RandomState(1).choice(len(_Xtr), N_MARG, replace=False)]; _SC = make_scheme_data(_Xtr, _pr, _Rm)
_x0 = _Xin[0]; _Dm = gower_matrix(_SC, _x0); _rng = np.random.RandomState(7); _first = np.r_[_j, [v for v in range(len(VARS)) if v != _j]]; _last = np.r_[[v for v in range(len(VARS)) if v != _j], _j]
_res = faith_child(_ft, _x0, float(_ft(_x0[None, :])[0]), _Dm, _SC, [_first, _last], [_rng.permutation(len(VARS)) for _ in range(N_RAND * 2)])
for _sch in ("cond", "marg"):
    _D, _Dr = _res[_sch][0], _res[_sch][1]
    assert _D[0] > _Dr > _D[1], f"U5: deletion score of the informative-first order must exceed the random mean, which must exceed informative-last ({_sch}: {_D}, {_Dr})"
UT["U5_toy_first_rand_last_cond"] = [float(v) for v in (_res["cond"][0][0], _res["cond"][1], _res["cond"][0][1])]
UT["U5_toy_first_rand_last_marg"] = [float(v) for v in (_res["marg"][0][0], _res["marg"][1], _res["marg"][0][1])]
# the reference rows of the deletion set must be disjoint from every explainer reference set (checked again per fold in the run)
WORK = dict(WORK) if isinstance(WORK, dict) else {}
WORK["unit_tests"] = time.time() - _t_unit; UNIT_TESTS_PASSED = True
print("unit tests passed", UT)

## M1.4 Runner
For every fold: fit LR / RF / MLP once (training seed `TS[0]`, identical to the main run), explain the **same children** as the main run under three arms, then score deletion-faithfulness for the ranking each explainer produced (explainer-seed run 0). Reference draws, deletion reference set and neighbour searches use the **training fold only**; the deletion reference set is disjoint from every explainer reference set.

In [ ]:
def run_m1(folds, n_per_fold, tag, verbose=True):
    cat_idx = [j for j, v in enumerate(VARS) if v in CAT_VARS]
    ATT, DIAG, FAITH, REAL, TIM = {}, [], [], [], {}
    def tick(key, t0, n=1):
        a = TIM.setdefault(key, [0.0, 0]); a[0] += time.time() - t0; a[1] += n
    def put(arm, k, m, meth, r, arrs): ATT[(arm, k, m, meth, r)] = arrs
    for k, (tr, te) in enumerate(folds):
        t_fold = time.time()
        prep = Prep(VARS, CAT_VARS).fit(X[tr]); Xtr = prep.impute(X[tr]); Ztr = prep.encode(Xtr); ytr = y[tr]
        sel = pick_instances(te, groups, n_per_fold, SEED_INST + k); Xin = prep.impute(X[sel]); Zin = prep.encode(Xin)
        refs = [make_refs(k, len(Xtr), r) for r in range(R_B)]
        used = set(np.concatenate([np.concatenate(t) for t in refs]).tolist())
        pool = np.array([i for i in range(len(Xtr)) if i not in used]); assert len(pool) >= N_MARG, "not enough training rows left for the deletion reference set"
        Rm = Xtr[np.random.RandomState(SEED_BG + 777000 + k).choice(pool, N_MARG, replace=False)]
        SC = make_scheme_data(Xtr, prep, Rm)
        assert not (set(np.random.RandomState(SEED_BG + 777000 + k).choice(pool, N_MARG, replace=False).tolist()) & used), "deletion reference set must be disjoint from explainer references"
        c1b, c2b = constraint_viol(Xtr)
        DIAG.append(dict(tag=tag, fold=k, model="all", check="constraint_C1_baseline_violation", value=float(c1b.mean())))
        DIAG.append(dict(tag=tag, fold=k, model="all", check="constraint_C2_baseline_violation", value=float(c2b.mean())))
        for m in MODELS:
            t0 = time.time(); fm = fit_model(m, Ztr, ytr, TS[0]); tick((m, "fit"), t0); f = make_predict_raw(prep, fm)
            meths = METHODS_BY_MODEL[m]
            # ---- M0: default configuration (reproduces the main run's first R_EM explainer seeds)
            if "SHAP" in meths:
                t0 = time.time(); outs = [shap_perm(f, Xtr, Xin, ES[e], SEED_BG + k) for e in range(R_EM)]; tick((m, "SHAP"), t0, len(Xin) * R_EM)
                put("M0", k, m, "SHAP", 0, [o[0] for o in outs]); DIAG += [dict(tag=tag, fold=k, model=m, arm="M0", check="shap_additivity_max_err", value=float(o[1].max())) for o in outs]
            if "LIME" in meths:
                t0 = time.time(); outs = [lime_run(f, Xtr, Xin, cat_idx, list(VARS), ES[e]) for e in range(R_EM)]; tick((m, "LIME"), t0, len(Xin) * R_EM)
                put("M0", k, m, "LIME", 0, [o[0] for o in outs]); DIAG += [dict(tag=tag, fold=k, model=m, arm="M0", check="lime_local_r2_mean", value=float(o[1].mean())) for o in outs]
            if "IG" in meths:
                t0 = time.time(); res, dls = ig_run(fm, prep, Zin, Ztr, Xtr, variants=False); tick((m, "IG"), t0, len(Xin))
                put("M0", k, m, "IG", 0, [res["IG"]]); DIAG.append(dict(tag=tag, fold=k, model=m, arm="M0", check="ig_delta_max", value=float(dls[0].max())))
            # ---- M1 (shared reference) and C1 (independent references), reference draws r = 0..R_B-1
            for r in range(R_B):
                bi, bpi, bppi = refs[r]; B, Bp = Xtr[bi], Xtr[bpi]; ZB, ZBpp = prep.encode(B), prep.encode(Xtr[bppi])
                t0 = time.time(); outs = [shap_with_bg(f, B, Xin, ES[e]) for e in range(R_EM)]; tick((m, "SHAP"), t0, len(Xin) * R_EM)
                put("M1", k, m, "SHAP", r, [o[0] for o in outs]); put("C1", k, m, "SHAP", r, [o[0] for o in outs])      # SHAP is identical in C1 (its reference is B)
                DIAG += [dict(tag=tag, fold=k, model=m, arm="M1", r=r, check="shap_additivity_max_err", value=float(o[1].max())) for o in outs]
                if "LIME" in meths:
                    t0 = time.time(); o1 = [lime_run(f, B, Xin, cat_idx, list(VARS), ES[e]) for e in range(R_EM)]; tick((m, "LIME"), t0, len(Xin) * R_EM)
                    t0 = time.time(); o2 = [lime_run(f, Bp, Xin, cat_idx, list(VARS), ES[e]) for e in range(R_EM)]; tick((m, "LIME"), t0, len(Xin) * R_EM)
                    put("M1", k, m, "LIME", r, [o[0] for o in o1]); put("C1", k, m, "LIME", r, [o[0] for o in o2])
                    DIAG += [dict(tag=tag, fold=k, model=m, arm="M1", r=r, check="lime_local_r2_mean", value=float(o[1].mean())) for o in o1]
                    DIAG += [dict(tag=tag, fold=k, model=m, arm="C1", r=r, check="lime_local_r2_mean", value=float(o[1].mean())) for o in o2]
                if "IG" in meths:
                    t0 = time.time(); a1, d1, e1 = ig_run_B(fm, prep, Zin, ZB); a2, d2, e2 = ig_run_B(fm, prep, Zin, ZBpp); tick((m, "IG"), t0, 2 * len(Xin))
                    put("M1", k, m, "IG", r, [a1]); put("C1", k, m, "IG", r, [a2])
                    DIAG += [dict(tag=tag, fold=k, model=m, arm="M1", r=r, check="ig_delta_max", value=d1), dict(tag=tag, fold=k, model=m, arm="C1", r=r, check="ig_delta_max", value=d2),
                             dict(tag=tag, fold=k, model=m, arm="M1", r=r, check="ig_avg_completeness_err", value=e1), dict(tag=tag, fold=k, model=m, arm="C1", r=r, check="ig_avg_completeness_err", value=e2)]
            # ---- deletion-faithfulness of every ranking (explainer-seed run 0)
            t0 = time.time(); meta, ords = [], []
            for arm in ("M0", "M1", "C1"):
                for meth in meths:
                    for r in (range(1) if arm == "M0" else range(R_B)):
                        a = np.abs(ATT[(arm, k, m, meth, r)][0])
                        meta.append((arm, meth, r, False)); ords.append(np.argsort(-a, axis=1, kind="stable"))
                        if arm == "M0": meta.append((arm, meth, r, True)); ords.append(np.argsort(a, axis=1, kind="stable"))         # reverse order, context only
            f0 = fbatch(f, Xin)
            for i in range(len(sel)):
                Dm = gower_matrix(SC, Xin[i]); rr = np.random.RandomState(SEED_NOISE + 1000 * k + i); rand_orders = [rr.permutation(len(VARS)) for _ in range(N_RAND)]
                res = faith_child(f, Xin[i], float(f0[i]), Dm, SC, [o[i] for o in ords], rand_orders)
                for scheme in ("cond", "marg"):
                    Dv, Dr = res[scheme]
                    for j, (arm, meth, r, rev) in enumerate(meta):
                        FAITH.append(dict(tag=tag, fold=k, model=m, child=i, scheme=scheme, arm=arm, meth=meth, r=r, reverse=rev, D=float(Dv[j]), D_rand=Dr))
                if m == "LR" and i < N_REAL_CHILD:                          # realism of the composite rows does not depend on the model
                    for row in realism_diag(Xin[i], Dm, SC, SEED_NOISE + 5000 + 100 * k + i): REAL.append(dict(tag=tag, fold=k, child=i, **row))
            tick((m, "faith"), t0, len(sel))
        if verbose: print(f"[{tag}] fold {k} done in {time.time() - t_fold:5.0f}s  (explained {len(sel)} children)")
    return dict(att=ATT, diag=pd.DataFrame(DIAG), faith=pd.DataFrame(FAITH), real=pd.DataFrame(REAL), timers=TIM, tag=tag, n_children=n_per_fold * len(folds))

## M1.5 Stage gate 1 — compute projection
Runs fold 0 with 5 children under the full M1 configuration (all reference draws, all explainer seeds) and scales to the configured MODE. Hard cap: `max_projected_hours` (specification: 3 h). If the cap is exceeded, reduce `R_B` (5 → 3) **before** the main run; never drop M0 or M1.

In [ ]:
t_probe = time.time()
PROBE = run_m1(FOLDS[:1], 5, "probe", verbose=False)
_dt = time.time() - t_probe
proj_h = (_dt / 5 * N_PER_FOLD * N_OUTER_FOLDS) * 1.15 / 3600 + _dt / 3600          # scale per-child work, + 15% for analysis overhead, + the probe itself
WORK["probe"] = _dt
print(f"probe {_dt:.0f}s | projected run time for MODE={MODE}: {proj_h:.2f} h (cap {M1_GATES['max_projected_hours']} h)")
print({f"{m}/{kk}": (f"{sec / max(n, 1):.3f}s per unit" if kk != "fit" else f"{sec:.1f}s per fit") for (m, kk), (sec, n) in PROBE["timers"].items()})
assert proj_h <= M1_GATES["max_projected_hours"], "Projected run time exceeds the specified cap: reduce R_B (5 -> 3) BEFORE the main run."

## M1.6 MAIN RUN

In [ ]:
t_main = time.time()
M1RUN = run_m1(FOLDS, N_PER_FOLD, "m1")
WORK["main"] = time.time() - t_main
print(f"M1 run finished in {(time.time() - t_main) / 60:.1f} min; faithfulness rows: {len(M1RUN['faith'])}")

## M1.7 Analysis (decision rules are the specification's; nothing here was tuned on results)
**Primary outcome** (Spearman, 7 cells): Δ_i = (C_i − N2_i)[M1] − (C_i − N2_i)[M0] per explained child, paired bootstrap over children (conditional on fitted folds). "Reduced" needs mean Δ ≥ 0.05, the Bonferroni-adjusted interval (level 1 − 0.05/7) above 0, **and** faithfulness F1 and F2 for both explainers of the pair (conditional deletion). `M1 − C1` isolates the effect of *sharing* the reference. Other metrics are descriptive.

In [ ]:
ATT = M1RUN["att"]; nf = len(FOLDS)
def arm_rs(arm): return [0] if arm == "M0" else list(range(R_B))
def runs(arm, k, m, meth, r): return ATT[(arm, k, m, meth, r)]
def cat_folds(fn_over_k):
    parts = [fn_over_k(k) for k in range(nf)]; return np.concatenate(parts), np.concatenate([np.full(len(p), k) for k, p in enumerate(parts)])

CROSS = {}                      # (arm, m, A, B, metric) -> dict(C, N2, Cr, N2r, fold)
for arm in ("M0", "M1", "C1"):
    for (m, A, B) in CELLS:
        for mname, fn in METRICS.items():
            Cr, N2r = [], []
            for r in arm_rs(arm):
                C, fold = cat_folds(lambda k: mean_pairs(runs(arm, k, m, A, r), runs(arm, k, m, B, r), fn, False))
                N2, _ = cat_folds(lambda k: n2_child(runs(arm, k, m, A, r)[0], runs(arm, k, m, B, r)[0], fn))
                Cr.append(C); N2r.append(N2)
            CROSS[(arm, m, A, B, mname)] = dict(C=np.mean(Cr, 0), N2=np.mean(N2r, 0), Cr=Cr, N2r=N2r, fold=fold)

def boot_ci(v, B_=2000, seed=SEED_BOOT, alpha_adj=0.05 / N_CELLS):
    v = np.asarray(v, float); v = v[np.isfinite(v)]; n = len(v); rng = np.random.RandomState(seed); mb = v[rng.randint(0, n, (B_, n))].mean(1)
    p = float(min(1.0, 2 * min((mb <= 0).mean(), (mb >= 0).mean())))
    return dict(mean=float(v.mean()), lo95=float(np.percentile(mb, 2.5)), hi95=float(np.percentile(mb, 97.5)),
                lo_adj=float(np.percentile(mb, 100 * alpha_adj / 2)), hi_adj=float(np.percentile(mb, 100 * (1 - alpha_adj / 2))), p_boot=p, n=n)

# ---- U4: M0 reproduces the FULL run (only meaningful with the cohort of the main run)
REPRO = []
if len(y) == 4105 and int(y.sum()) == 966 and MODE != "SMOKE":
    for (m, A, B) in CELLS:
        c = float(CROSS[("M0", m, A, B, "Spearman")]["C"].mean()); REPRO.append(dict(kind="cross_C", cell=f"{m} {A}-{B}", m1_run=c, full_run=FULL_REF_C[(m, A, B)], abs_diff=abs(c - FULL_REF_C[(m, A, B)]), tol=M1_GATES["repro_tol_C"]))
    for (m, meth), ref in FULL_REF_FLOOR_E.items():
        fl = float(np.concatenate([mean_pairs(runs("M0", k, m, meth, 0), runs("M0", k, m, meth, 0), sim_spearman, True) for k in range(nf)]).mean())
        REPRO.append(dict(kind="floorE", cell=f"{m} {meth}", m1_run=fl, full_run=ref, abs_diff=abs(fl - ref), tol=M1_GATES["repro_tol_floor"]))
REPRO = pd.DataFrame(REPRO)
if len(REPRO): print(REPRO.round(4).to_string(index=False))

# ---- primary table: Delta for M1 vs M0, C1 vs M0 and sharing effect M1 vs C1, all four metrics
rows = []
for (m, A, B) in CELLS:
    for mname in METRICS:
        cm0, cm1, cc1 = CROSS[("M0", m, A, B, mname)], CROSS[("M1", m, A, B, mname)], CROSS[("C1", m, A, B, mname)]
        base = cm0["C"] - cm0["N2"]; x1 = cm1["C"] - cm1["N2"]; xc = cc1["C"] - cc1["N2"]
        for label, delta, arm_r in (("M1_vs_M0", x1 - base, "M1"), ("C1_vs_M0", xc - base, "C1"), ("M1_vs_C1", x1 - xc, None)):
            ci = boot_ci(delta)
            per_r = [float(np.nanmean(((CROSS[(arm_r, m, A, B, mname)]["Cr"][r] - CROSS[(arm_r, m, A, B, mname)]["N2r"][r]) - base))) for r in range(R_B)] if arm_r else []
            fold = cm0["fold"]; per_fold = [float(np.nanmean(delta[fold == k])) for k in range(nf)]
            rows.append(dict(model=m, pair=f"{A} vs {B}", metric=mname, contrast=label, **ci, per_draw_means=json.dumps([round(v, 4) for v in per_r]), per_fold_means=json.dumps([round(v, 4) for v in per_fold]),
                             C_M0=float(cm0["C"].mean()), C_M1=float(cm1["C"].mean()), C_C1=float(cc1["C"].mean()), N2_M0=float(cm0["N2"].mean()), N2_M1=float(cm1["N2"].mean()), N2_C1=float(cc1["N2"].mean())))
DELTA = pd.DataFrame(rows)
print(DELTA[(DELTA.metric == "Spearman")][["model", "pair", "contrast", "mean", "lo95", "hi95", "lo_adj", "hi_adj", "C_M0", "C_M1", "C_C1", "N2_M0", "N2_M1"]].round(3).to_string(index=False))

# ---- faithfulness table
FA = M1RUN["faith"].copy(); FA["A"] = FA["D"] - FA["D_rand"]
def child_key(df): return df.fold.astype(int) * 100000 + df.child.astype(int)
fa_rows, FAITH_OK = [], {}
for scheme in ("cond", "marg"):
    for m in MODELS:
        for meth in METHODS_BY_MODEL[m]:
            sub = FA[(FA.scheme == scheme) & (FA.model == m) & (FA.meth == meth) & (~FA.reverse)]
            def per_child(arm):
                s = sub[sub.arm == arm].copy(); s["ck"] = child_key(s); return s.groupby("ck")["A"].mean()
            a0, a1, ac = per_child("M0"), per_child("M1"), per_child("C1"); idx = a0.index
            a1, ac = a1.reindex(idx), ac.reindex(idx)
            rev = FA[(FA.scheme == scheme) & (FA.model == m) & (FA.meth == meth) & (FA.reverse) & (FA.arm == "M0")]; rev = rev.assign(ck=child_key(rev)).groupby("ck")["D"].mean().reindex(idx)
            d0 = sub[sub.arm == "M0"].assign(ck=lambda d: child_key(d)).groupby("ck")["D"].mean().reindex(idx); dr = sub[sub.arm == "M0"].assign(ck=lambda d: child_key(d)).groupby("ck")["D_rand"].mean().reindex(idx)
            c0, c1, cc = boot_ci(a0.values), boot_ci(a1.values), boot_ci(ac.values); dd = boot_ci((a1 - a0).values)
            F1 = {arm: bool(c["lo95"] > 0) for arm, c in (("M0", c0), ("M1", c1), ("C1", cc))}
            F2 = bool(c0["mean"] > 0 and dd["lo95"] > -FAITH_MARGIN * c0["mean"])
            fa_rows.append(dict(scheme=scheme, model=m, explainer=meth, A_M0=c0["mean"], A_M0_lo=c0["lo95"], A_M1=c1["mean"], A_M1_lo=c1["lo95"], A_C1=cc["mean"], A_C1_lo=cc["lo95"],
                                diff_M1_minus_M0=dd["mean"], diff_lo95=dd["lo95"], F1_M0=F1["M0"], F1_M1=F1["M1"], F1_C1=F1["C1"], F2_M1_vs_M0=F2, uninformative=bool(c0["mean"] <= 0),
                                D_M0=float(d0.mean()), D_random=float(dr.mean()), D_reverse_M0=float(rev.mean()), frac_children_better_than_random_M0=float((a0 > 0).mean())))
            FAITH_OK[(scheme, m, meth)] = bool(F1["M1"] and F2)
FAITHTAB = pd.DataFrame(fa_rows)
print(FAITHTAB[["scheme", "model", "explainer", "A_M0", "A_M1", "A_C1", "diff_M1_minus_M0", "F1_M1", "F2_M1_vs_M0", "uninformative", "D_M0", "D_random", "D_reverse_M0"]].round(4).to_string(index=False))

# ---- decision per cell (Spearman, M1 vs M0), specification rule
dec = []
for (m, A, B) in CELLS:
    r = DELTA[(DELTA.model == m) & (DELTA.pair == f"{A} vs {B}") & (DELTA.metric == "Spearman") & (DELTA.contrast == "M1_vs_M0")].iloc[0]
    faith_cond = FAITH_OK[("cond", m, A)] and FAITH_OK[("cond", m, B)]; faith_marg = FAITH_OK[("marg", m, A)] and FAITH_OK[("marg", m, B)]
    unin = bool(FAITHTAB[(FAITHTAB.scheme == "cond") & (FAITHTAB.model == m) & (FAITHTAB.explainer.isin([A, B]))].uninformative.any())
    if r["mean"] >= DELTA_MIN and r["lo_adj"] > 0: verdict = ("REDUCED and faithful" if faith_cond else ("REDUCED but faithfulness uninformative" if unin else "REDUCED but faithfulness NOT preserved"))
    elif r["mean"] <= -DELTA_MIN and r["hi_adj"] < 0: verdict = "INCREASED"
    else: verdict = "no clear change"
    dec.append(dict(model=m, pair=f"{A} vs {B}", delta_mean=r["mean"], ci_adj=f"[{r['lo_adj']:.3f}, {r['hi_adj']:.3f}]", ci95=f"[{r['lo95']:.3f}, {r['hi95']:.3f}]", faithful_conditional=faith_cond, faithful_marginal=faith_marg, verdict=verdict))
DECISION = pd.DataFrame(dec); print(DECISION.to_string(index=False))

# ---- secondary: floors (E axis; reference-draw axis for M1/C1) and excess disagreement (NOT like-for-like across arms: see specification)
sec = []
for arm in ("M0", "M1", "C1"):
    for (m, A, B) in CELLS:
        mname = "Spearman"; cr = CROSS[(arm, m, A, B, mname)]
        def floorE(meth):
            vals = []
            for r in arm_rs(arm):
                E = [runs(arm, k, m, meth, r) for k in range(nf)]
                vals.append(np.concatenate([mean_pairs(e, e, sim_spearman, True) if len(e) > 1 else np.ones(len(e[0])) for e in E]))
            return np.mean(vals, 0)
        def floorB(meth):
            if arm == "M0": return None
            per_k = []
            for k in range(nf):
                Rs = [runs(arm, k, m, meth, r)[0] for r in range(R_B)]; per_k.append(mean_pairs(Rs, Rs, sim_spearman, True))
            return np.concatenate(per_k)
        FEa, FEb = floorE(A), floorE(B); ok = ~np.isnan(cr["N2"])
        _, kap, exc = kappa_parts(cr["C"][ok].mean(), FEa[ok].mean(), FEb[ok].mean(), cr["N2"][ok].mean())
        row = dict(arm=arm, model=m, pair=f"{A} vs {B}", C=float(cr["C"][ok].mean()), N2=float(cr["N2"][ok].mean()), floorE_A=float(FEa[ok].mean()), floorE_B=float(FEb[ok].mean()), kappa_Eonly=float(kap), excess_Eonly=float(exc))
        if arm != "M0":
            fa_, fb_ = floorB(A), floorB(B); cfa, cfb = np.nanmin(np.vstack([FEa, fa_]), 0), np.nanmin(np.vstack([FEb, fb_]), 0)
            _, kap2, exc2 = kappa_parts(cr["C"][ok].mean(), cfa[ok].mean(), cfb[ok].mean(), cr["N2"][ok].mean())
            row.update(floorB_A=float(fa_[ok].mean()), floorB_B=float(fb_[ok].mean()), kappa_E_and_B=float(kap2), excess_E_and_B=float(exc2))
        sec.append(row)
SECOND = pd.DataFrame(sec); print(SECOND.round(3).to_string(index=False))

## M1.8 Realism diagnostics (descriptive) and LIME local fit
Share of composite rows that violate the two logical constraints between mother's age and the child's age, and the typical Gower distance from a composite row to the nearest real training child (conditional vs marginal deletion), next to the same distance for the real explained children. The constraint **baseline** (violation rate among real training rows) is shown first: a constraint with a baseline above 2% is not informative and is reported as such.

In [ ]:
DG = M1RUN["diag"].copy()
base_viol = DG[DG.check.str.startswith("constraint_")].groupby("check")["value"].mean()
print("baseline constraint violation among real training rows:", base_viol.round(4).to_dict())
REALTAB = M1RUN["real"].groupby(["scheme", "k"])[["viol_C1", "viol_C2", "median_nn_gower", "real_child_nn_gower"]].mean().reset_index()
print(REALTAB.round(4).to_string(index=False))
LIME_R2 = DG[DG.check == "lime_local_r2_mean"].groupby(["model", "arm"])["value"].mean().unstack().round(3); print("LIME local R^2 by arm:\n", LIME_R2)

## M1.9 Gate report (specification section 5)

In [ ]:
gates = []
def gate(name, ok, detail): gates.append(dict(gate=name, passed=bool(ok), detail=str(detail)))
gate("GM0 unit tests U1/U2/U5 ran and passed in this session", UNIT_TESTS_PASSED, UT)
gate("GM1 compute cap (projection from the probe AND measured work)", proj_h <= M1_GATES["max_projected_hours"] and (WORK["probe"] + WORK["main"]) / 3600 <= M1_GATES["max_projected_hours"], f"projected {proj_h:.2f} h; measured {(WORK['probe'] + WORK['main']) / 3600:.2f} h")
if len(REPRO): gate("GM2 / U4 M0 reproduces the FULL-run Spearman cross-method C and E-axis floors", bool((REPRO.abs_diff <= REPRO.tol).all()), f"max |diff| C {REPRO[REPRO.kind == 'cross_C'].abs_diff.max():.4f} (tol {M1_GATES['repro_tol_C']}), floors {REPRO[REPRO.kind == 'floorE'].abs_diff.max():.4f} (tol {M1_GATES['repro_tol_floor']})")
else: gate("GM2 / U4 M0 reproduction (FULL/PILOT cohort only)", True, "not applicable in SMOKE mode or for a different cohort")
_shap_err = DG[DG.check == "shap_additivity_max_err"].value.max(); _ig = DG[DG.check == "ig_delta_max"].value.max() if (DG.check == "ig_delta_max").any() else 0.0
_ig_avg = DG[DG.check == "ig_avg_completeness_err"].value.max() if (DG.check == "ig_avg_completeness_err").any() else 0.0
gate("GM3a SHAP additivity (all arms)", _shap_err < M1_GATES["max_additivity_err"], f"max err {_shap_err:.2e}")
gate("GM3b IG completeness: per-baseline delta (M0: mean baseline; M1/C1: every real baseline) and the averaged attribution", _ig < M1_GATES["max_ig_delta"] and _ig_avg < M1_GATES["max_ig_delta"], f"max per-baseline |delta| {_ig:.2e}; max averaged-attribution error {_ig_avg:.2e}")
_drop = (LIME_R2["M0"] - LIME_R2[["M1", "C1"]].min(axis=1)).max()
gate("GM4 LIME local R^2 does not drop by more than 0.10 under a 50-row reference (reported, not a failure of the study)", _drop <= M1_GATES["max_lime_r2_drop"], f"max drop {_drop:.3f}")
_rev_bad = FAITHTAB[(FAITHTAB.scheme == "cond") & (FAITHTAB.D_M0 < FAITHTAB.D_reverse_M0)][["model", "explainer"]].values.tolist()
gate("GM5 M0 explainer order is not worse than the reverse order (conditional deletion)", not _rev_bad, f"violations: {_rev_bad}")
gate("GM6 logical-constraint baseline among real training rows <= 2% (else the constraint is uninformative)", bool((base_viol <= M1_GATES["max_constraint_baseline_viol"]).all()), base_viol.round(4).to_dict())
GATES_M1 = pd.DataFrame(gates); print(GATES_M1.to_string(index=False))

## M1.10 Save outputs (no IDs, no raw rows)

In [ ]:
import platform
from importlib import metadata as _md
def _v(p):
    try: return _md.version(p)
    except Exception: return "n/a"
meta = dict(notebook="m1_shared_reference_v1", mode=MODE, feature_set=FEATURE_SET, vars=VARS, seeds=dict(split=SEED_SPLIT, inst=SEED_INST, noise=SEED_NOISE, boot=SEED_BOOT, bg=SEED_BG, ES=ES[:R_EM], TS0=TS[0]),
            m1_settings=dict(R_B=R_B, R_EM=R_EM, K_REF=K_REF, K_NN=K_NN, N_MARG=N_MARG, N_RAND=N_RAND, DEL_K=DEL_K, DELTA_MIN=DELTA_MIN, FAITH_MARGIN=FAITH_MARGIN, gates=M1_GATES),
            n_per_fold=N_PER_FOLD, n_perm=N_PERM, lime_samples=LIME_SAMPLES, ig_steps=IG_STEPS, hp=HP, n_analytic=int(len(y)), n_stunted=int(y.sum()), n_clusters=int(len(np.unique(groups))),
            fold_sha256=FOLD_SHA, y_sha256=Y_SHA, x_sha256=X_SHA, expected_fold_sha256=EXPECTED_FOLD_SHA256, expected_x_sha256=EXPECTED_X_SHA256,
            overrides=dict(version_drift=ALLOW_VERSION_DRIFT, hash_mismatch=ALLOW_HASH_MISMATCH, strict=STRICT), unit_test_values=UT,
            work_seconds={k: float(v) for k, v in WORK.items()}, projected_hours=float(proj_h),
            versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, scipy=_v("scipy"), torch=torch.__version__, shap=_v("shap"), lime=_v("lime"), captum=_v("captum")))
# per-child primary values (fold and within-fold position only; no IDs)
pc = []
for (m, A, B) in CELLS:
    cm0, cm1, cc1 = (CROSS[(arm, m, A, B, "Spearman")] for arm in ("M0", "M1", "C1"))
    pc.append(pd.DataFrame(dict(model=m, pair=f"{A} vs {B}", fold=cm0["fold"], C_M0=cm0["C"], N2_M0=cm0["N2"], C_M1=cm1["C"], N2_M1=cm1["N2"], C_C1=cc1["C"], N2_C1=cc1["N2"])))
for name, df in (("m1_delta_all_metrics.csv", DELTA), ("m1_decision.csv", DECISION), ("m1_faithfulness.csv", FAITHTAB), ("m1_secondary_floors_excess.csv", SECOND), ("m1_gate_report.csv", GATES_M1),
                 ("m1_realism_diagnostics.csv", REALTAB), ("m1_lime_r2_by_arm.csv", LIME_R2.reset_index()), ("m1_reproduction_vs_full.csv", REPRO), ("m1_per_child_primary.csv", pd.concat(pc, ignore_index=True)),
                 ("m1_faithfulness_rows.csv", M1RUN["faith"]), ("m1_diagnostics.csv", DG)):
    df.to_csv(os.path.join(OUT_DIR, name), index=False)
with open(os.path.join(OUT_DIR, "m1_run_metadata.json"), "w") as fh: json.dump(meta, fh, indent=1, default=str)
print("saved to", OUT_DIR, sorted(os.listdir(OUT_DIR)))
try:
    import shutil; from google.colab import files
    shutil.make_archive("/content/m1_outputs", "zip", OUT_DIR); files.download("/content/m1_outputs.zip")
except Exception as e:
    print("zip/download skipped:", type(e).__name__)